# Agent with memory using Walrus Memory


This notebook is a runnable end-user walkthrough for the Walrus Memory Python SDK.

It uses the Python package/import name `memwal`, while the user-facing product name is Walrus Memory. The flow follows a small customer-support assistant that remembers a support conversation, waits for the memory job to persist, recalls the context, and then tours the SDK features that are useful in notebooks and application backends.

You will run:

- installation
- secure Colab Secrets or hidden prompts
- `MemWalSync.create(...)` for notebook-friendly calls
- relayer `health()` and `compatibility()`
- delegate public-key/address derivation without printing private keys
- `remember(...)` plus `wait_for_remember_job(...)`
- `recall(RecallParams(...))`
- `remember_and_wait(...)`
- bulk remember helpers
- optional `ask(...)`, `analyze(...)`, `analyze_and_wait(...)`, `embed(...)`, manual recall/register with scoring weights, and `restore(...)`
- optional OpenAI and LangChain middleware examples
- cleanup and troubleshooting

The notebook defaults to `staging` for test credentials. You can switch to `prod` when you have production credentials.


## Requirements

Install the SDK from PyPI. The Colab runtime may show dependency resolver output; that is fine as long as `memwal` installs successfully.


In [ ]:
%pip install -q -U memwal


## Configure Secrets

Create or open a Walrus Memory account and delegate key from the hosted dashboard for the environment you want to use.

In Colab, the safest path is the key icon in the left sidebar. Add these secrets:

- `MEMWAL_ACCOUNT_ID`
- `MEMWAL_PRIVATE_KEY`
- optional `WALRUS_MEMORY_ENV` with `staging` or `prod`
- optional `WALRUS_MEMORY_NAMESPACE`
- optional `RUN_OPTIONAL_SDK_METHODS`, set to `false` to skip slower optional Walrus Memory SDK cells
- optional `RUN_MIDDLEWARE_DEMO`, set to `true` only if you also want to run the OpenAI/LangChain wrapper cells
- optional `OPENAI_API_KEY`, `OPENAI_BASE_URL`, and `OPENAI_MODEL` for the middleware cells

If a value is missing from Colab Secrets, the next cell asks through a hidden prompt. Do not paste real secrets into notebook source cells or commit them to Git.

For OpenAI-compatible providers such as OpenRouter, set `OPENAI_BASE_URL` to the provider API base URL and set `OPENAI_MODEL` to that provider's model id.


In [ ]:
import getpass
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

ALLOWED_ENVS = {"staging", "prod"}
DEFAULT_ENV = "staging"


def read_secret(name: str, prompt: str, required: bool = True) -> str:
    value = os.environ.get(name, "")
    if not value and userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    if not value and required:
        value = getpass.getpass(prompt)
    if value:
        os.environ[name] = value
    return value


def read_setting(name: str, default: str) -> str:
    value = os.environ.get(name, "")
    if not value and userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    value = (value or default).strip()
    os.environ[name] = value
    return value


def mask(value: str, keep: int = 8) -> str:
    if not value:
        return "<missing>"
    if len(value) <= keep * 2:
        return value[:2] + "..."
    return f"{value[:keep]}...{value[-keep:]}"


def as_bool(value: str) -> bool:
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


read_secret("MEMWAL_PRIVATE_KEY", "Paste Walrus Memory delegate private key (hidden): ")
read_secret("MEMWAL_ACCOUNT_ID", "Paste Walrus Memory account ID (hidden): ")

WALRUS_MEMORY_ENV = read_setting("WALRUS_MEMORY_ENV", DEFAULT_ENV).lower()
if WALRUS_MEMORY_ENV not in ALLOWED_ENVS:
    raise ValueError("WALRUS_MEMORY_ENV must be 'staging' or 'prod' for this notebook.")
os.environ["WALRUS_MEMORY_ENV"] = WALRUS_MEMORY_ENV

DEFAULT_NAMESPACE = f"colab-{WALRUS_MEMORY_ENV}-customer-service-test"
WALRUS_MEMORY_NAMESPACE = read_setting("WALRUS_MEMORY_NAMESPACE", DEFAULT_NAMESPACE)
RUN_OPTIONAL_SDK_METHODS = as_bool(read_setting("RUN_OPTIONAL_SDK_METHODS", "true"))
RUN_MIDDLEWARE_DEMO = as_bool(read_setting("RUN_MIDDLEWARE_DEMO", "false"))
OPENAI_BASE_URL = read_setting("OPENAI_BASE_URL", "")
OPENAI_MODEL = read_setting("OPENAI_MODEL", "gpt-4o-mini")

print("Walrus Memory configuration loaded")
print("account_id:", mask(os.environ.get("MEMWAL_ACCOUNT_ID", "")))
print("env:", WALRUS_MEMORY_ENV)
print("namespace:", WALRUS_MEMORY_NAMESPACE)
print("optional SDK method cells:", RUN_OPTIONAL_SDK_METHODS)
print("optional middleware cells:", RUN_MIDDLEWARE_DEMO)
print("OpenAI-compatible base URL:", OPENAI_BASE_URL or "default OpenAI API")
print("OpenAI model for optional middleware:", OPENAI_MODEL)


## Create The Client

`MemWalSync` is the notebook-friendly wrapper. It exposes the same API surface as the async `MemWal` client, but you do not need `await` in Colab cells.

This cell also checks relayer health, validates SDK compatibility, derives the delegate public key, and derives the delegate Sui address locally. It masks public identifiers and never prints the private key.


In [ ]:
from memwal import (
    MemWalSync,
    RecallParams,
    RecallManualOptions,
    RememberBulkItem,
    RememberBulkOptions,
    RememberManualOptions,
    ScoringWeights,
    delegate_key_to_sui_address,
)

client_kwargs = {
    "key": os.environ["MEMWAL_PRIVATE_KEY"],
    "account_id": os.environ["MEMWAL_ACCOUNT_ID"],
    "namespace": WALRUS_MEMORY_NAMESPACE,
    "env": WALRUS_MEMORY_ENV,
}

memory = MemWalSync.create(**client_kwargs)


def memwal_call(action):
    # Keeps older memwal releases notebook-safe when multiple sync calls reuse the client.
    inner = getattr(memory, "_inner", None)
    if inner is not None:
        inner._client = None
    return action()


def show_error(name: str, exc: Exception) -> None:
    print(f"{name} skipped: {type(exc).__name__}: {str(exc)[:220]}")


health = memwal_call(memory.health)
print(f"relayer status={health.status} version={health.version}")

compatibility = memwal_call(memory.compatibility)
print("api version:", compatibility.get("apiVersion") or compatibility.get("api_version") or "unknown")
print("min python sdk:", (compatibility.get("minSupportedSdk") or {}).get("python", "unknown"))

public_key_hex = memwal_call(memory.get_public_key_hex)
delegate_address = delegate_key_to_sui_address(os.environ["MEMWAL_PRIVATE_KEY"])
print("delegate public key:", mask(public_key_hex))
print("delegate Sui address:", mask(delegate_address))


## Build A Support Conversation

The example memory is a realistic customer-support exchange. It contains an issue, a product, an email, an order number, and an appointment window. The namespace isolates this demo from your application data.


In [ ]:
conversation = [
    {
        "role": "assistant",
        "content": "Hi, I am the support chatbot. Thanks for being a member. What can I help you with?",
    },
    {
        "role": "user",
        "content": "I am seeing horizontal lines on my TV. Model: Sony 77 inch BRAVIA XR A80K OLED 4K UHD Smart TV.",
    },
    {
        "role": "assistant",
        "content": "I can help check warranty and repair options. Can you confirm the email on the account?",
    },
    {"role": "user", "content": "demo.customer@example.com"},
    {
        "role": "assistant",
        "content": "Could you share the order number so I can look it up quickly?",
    },
    {"role": "user", "content": "Order number: 112217629"},
    {
        "role": "assistant",
        "content": "The order is in warranty. A technician appointment is available tomorrow from 9 AM to noon.",
    },
]

conversation_text = "Customer service conversation:\n" + "\n".join(
    f"{message['role']}: {message['content']}" for message in conversation
)

print(conversation_text)


## Remember And Wait For Persistence

`remember(...)` returns quickly with a job id. The relayer then encrypts, uploads to Walrus, and indexes in the background. Use `wait_for_remember_job(...)` when you need proof that the memory is persisted before continuing.


In [ ]:
accepted = memwal_call(lambda: memory.remember(conversation_text))
print(f"remember accepted: job_id={accepted.job_id} status={accepted.status}")

stored = memwal_call(lambda: memory.wait_for_remember_job(accepted.job_id, timeout_ms=120_000))
print("remember job status=done")
print(f"stored: blob_id={stored.blob_id[:24]}... namespace={stored.namespace}")


## Recall The Support Context

`recall(RecallParams(...))` performs semantic search inside your account and namespace. Lower distance means stronger similarity; relevance below is displayed as `1 - distance` for readability.


In [ ]:
question = "What order number and service appointment did the customer already discuss?"
relevant_memories = memwal_call(
    lambda: memory.recall(
        RecallParams(
            query=question,
            limit=5,
            namespace=WALRUS_MEMORY_NAMESPACE,
            max_distance=0.85,
        )
    )
)

print(f"recall returned {len(relevant_memories.results)} result(s)")
for index, item in enumerate(relevant_memories.results, start=1):
    relevance = 1 - item.distance
    print(f"\nResult {index} relevance={relevance:.2f}")
    print(item.text[:900])


## One-Call Persistence With `remember_and_wait`

Use `remember_and_wait(...)` when you do not need the accepted-job phase separately.


In [ ]:
preference_text = (
    "Customer service preference: The customer prefers SMS updates, morning appointments, "
    "and warranty-first troubleshooting for TV repair requests."
)

preference = memwal_call(
    lambda: memory.remember_and_wait(preference_text, timeout_ms=120_000)
)
print(f"preference stored: blob_id={preference.blob_id[:24]}... namespace={preference.namespace}")


## Bulk Remember

Bulk APIs are useful when you import several small facts at once. The accepted result returns one job id per item; then you can poll all jobs together.


In [ ]:
bulk_items = [
    RememberBulkItem(text="Support fact: customer owns a Sony BRAVIA XR A80K TV."),
    RememberBulkItem(text="Support fact: customer order number is 112217629."),
    RememberBulkItem(text="Support fact: available appointment window is tomorrow from 9 AM to noon."),
]

bulk = memwal_call(lambda: memory.remember_bulk(bulk_items))
print(f"bulk accepted: total={bulk.total} status={bulk.status}")
print("bulk job ids:", ", ".join(job_id[:8] + "..." for job_id in bulk.job_ids))

initial_status = memwal_call(lambda: memory.get_remember_bulk_status(bulk.job_ids))
print("initial statuses:", [item.status for item in initial_status.results])

bulk_done = memwal_call(
    lambda: memory.wait_for_remember_jobs(
        bulk.job_ids,
        RememberBulkOptions(timeout_ms=120_000),
    )
)
print(
    f"bulk settled: total={bulk_done.total} "
    f"succeeded={bulk_done.succeeded} failed={bulk_done.failed} timed_out={bulk_done.timed_out}"
)


## One-Call Bulk Persistence

`remember_bulk_and_wait(...)` is the convenience form when you want to submit a batch and wait for every item to settle in one call. The explicit accepted-job aliases `remember_async(...)` and `remember_bulk_async(...)` mirror the same accepted-job behavior used by `remember(...)` and `remember_bulk(...)` in this notebook.


In [ ]:
followup_bulk = [
    RememberBulkItem(text="Support preference: customer wants SMS updates before technician arrival."),
    RememberBulkItem(text="Support preference: customer prefers morning service appointments."),
]

followup_done = memwal_call(
    lambda: memory.remember_bulk_and_wait(
        followup_bulk,
        RememberBulkOptions(timeout_ms=120_000),
    )
)
print(
    f"remember_bulk_and_wait settled: total={followup_done.total} "
    f"succeeded={followup_done.succeeded} failed={followup_done.failed} timed_out={followup_done.timed_out}"
)


## Ask From Memory

`ask(...)` is the highest-level retrieval helper. It recalls relevant memory and returns an answer with the number of memories used. This is optional because some hosted environments may gate answer synthesis.


In [ ]:
if RUN_OPTIONAL_SDK_METHODS:
    try:
        answer = memwal_call(lambda: memory.ask("When is the technician appointment?", limit=5))
        print("ask answer:", answer.answer)
        print("ask memories used:", answer.memories_used)
    except Exception as exc:
        show_error("ask", exc)
else:
    print("ask skipped. Set RUN_OPTIONAL_SDK_METHODS=true to run optional SDK methods.")


## Analyze Facts Without Waiting

`analyze(...)` extracts memorable facts and returns accepted remember job IDs. Use it when you want to inspect or manage the jobs yourself.


In [ ]:
if RUN_OPTIONAL_SDK_METHODS:
    try:
        accepted_analysis = memwal_call(
            lambda: memory.analyze(
                "Customer prefers SMS updates, warranty-first troubleshooting, and morning appointments."
            )
        )
        print(f"analyze accepted facts={accepted_analysis.fact_count} status={accepted_analysis.status}")
        print("analyze job ids:", ", ".join(job_id[:8] + "..." for job_id in accepted_analysis.job_ids))
        for fact in accepted_analysis.facts[:3]:
            print("-", fact.text)
    except Exception as exc:
        show_error("analyze", exc)
else:
    print("analyze skipped.")


## Analyze And Persist Facts

`analyze_and_wait(...)` asks the relayer to extract memorable facts from text and waits for the resulting remember jobs to settle. It is useful when you want the SDK to decide which facts should become memory.


In [ ]:
if RUN_OPTIONAL_SDK_METHODS:
    try:
        analyzed = memwal_call(
            lambda: memory.analyze_and_wait(
                "Customer says they prefer SMS updates and morning support appointments.",
                opts=RememberBulkOptions(timeout_ms=120_000),
            )
        )
        print(f"analyze facts={len(analyzed.facts)} succeeded={analyzed.succeeded}")
        for fact in analyzed.facts[:3]:
            print("-", fact.text)
    except Exception as exc:
        show_error("analyze_and_wait", exc)
else:
    print("analyze_and_wait skipped.")


## Embeddings And Manual Indexing

`embed(...)`, `remember_manual(...)`, and `recall_manual(...)` expose the lower-level vector path. Most users should prefer `remember(...)` and `recall(...)`, but the manual methods are useful when you already have a Walrus blob ID and want to register or search by a supplied vector.

This example also passes `ScoringWeights` to `recall_manual(...)`, which mirrors the same weighted-ranking option available in the TypeScript SDK.


In [ ]:
embedded_vector = None

if RUN_OPTIONAL_SDK_METHODS:
    try:
        embedded = memwal_call(lambda: memory.embed("warranty repair appointment order number"))
        embedded_vector = embedded.vector
        print("embedding dimensions:", len(embedded_vector))
    except Exception as exc:
        show_error("embed", exc)

    if embedded_vector:
        try:
            manual = memwal_call(
                lambda: memory.remember_manual(
                    RememberManualOptions(
                        blob_id=stored.blob_id,
                        vector=embedded_vector,
                        namespace=WALRUS_MEMORY_NAMESPACE,
                    )
                )
            )
            print(f"manual register: blob_id={manual.blob_id[:24]}... namespace={manual.namespace}")
        except Exception as exc:
            show_error("remember_manual", exc)

        try:
            manual_hits = memwal_call(
                lambda: memory.recall_manual(
                    RecallManualOptions(
                        vector=embedded_vector,
                        limit=5,
                        namespace=WALRUS_MEMORY_NAMESPACE,
                        scoring_weights=ScoringWeights(
                            semantic=1.0,
                            recency=0.1,
                            recency_half_life_days=7,
                            importance=0.0,
                        ),
                    )
                )
            )
            print(f"manual recall hits: {manual_hits.total}")
            for hit in manual_hits.results[:3]:
                print(f"- blob_id={hit.blob_id[:24]}... distance={hit.distance:.3f}")
        except Exception as exc:
            show_error("recall_manual", exc)
else:
    print("embed/manual indexing skipped.")


## Restore Index Rows

`restore(...)` asks the relayer to repair missing index rows from Walrus-backed memory for a namespace. It is safe to run with a small limit in a demo namespace.


In [ ]:
if RUN_OPTIONAL_SDK_METHODS:
    try:
        restored = memwal_call(lambda: memory.restore(WALRUS_MEMORY_NAMESPACE, limit=10))
        print(
            f"restore: restored={restored.restored} skipped={restored.skipped} "
            f"total={restored.total} namespace={restored.namespace}"
        )
    except Exception as exc:
        show_error("restore", exc)
else:
    print("restore skipped.")


## Optional OpenAI Middleware

`with_memwal_openai(...)` wraps an OpenAI SDK client. Before each chat call, it recalls relevant Walrus Memory context and injects it into the message list. After the call, it can analyze and save the latest user message.

This section is off by default because it installs an optional dependency, requires `OPENAI_API_KEY`, and sends the prompt plus recalled memory context to your configured model provider. For OpenAI-compatible providers such as OpenRouter, also set `OPENAI_BASE_URL` and the provider-specific `OPENAI_MODEL`. Set `RUN_MIDDLEWARE_DEMO=true` in Colab Secrets when you want to run it.


In [ ]:
if RUN_MIDDLEWARE_DEMO:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "memwal[openai]", "openai"])

    from openai import AsyncOpenAI
    from memwal import with_memwal_openai

    read_secret("OPENAI_API_KEY", "Paste OpenAI API key for middleware demo (hidden): ")

    openai_kwargs = {"api_key": os.environ["OPENAI_API_KEY"]}
    if OPENAI_BASE_URL:
        openai_kwargs["base_url"] = OPENAI_BASE_URL
    openai_client = AsyncOpenAI(**openai_kwargs)
    smart_openai = with_memwal_openai(
        openai_client,
        key=os.environ["MEMWAL_PRIVATE_KEY"],
        account_id=os.environ["MEMWAL_ACCOUNT_ID"],
        namespace=WALRUS_MEMORY_NAMESPACE,
        env=WALRUS_MEMORY_ENV,
        max_memories=5,
        min_relevance=0.2,
    )

    response = await smart_openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Use my support memory: what TV issue and appointment window should the agent remember?",
            }
        ],
        max_tokens=160,
    )
    print(response.choices[0].message.content)
else:
    print("OpenAI middleware demo skipped. Set RUN_MIDDLEWARE_DEMO=true to run it.")


## Optional LangChain Middleware

`with_memwal_langchain(...)` wraps a LangChain chat model with the same recall-before-call and save-after-call behavior. This is the Python SDK equivalent of adding memory to an existing agent or assistant stack without changing your chain logic.

Use the same `OPENAI_BASE_URL` and `OPENAI_MODEL` settings as the OpenAI middleware cell when running through an OpenAI-compatible provider.


In [ ]:
if RUN_MIDDLEWARE_DEMO:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "memwal[langchain]",
        "langchain-openai",
    ])

    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    from memwal import with_memwal_langchain

    read_secret("OPENAI_API_KEY", "Paste OpenAI API key for LangChain demo (hidden): ")

    llm_kwargs = {"model": OPENAI_MODEL, "api_key": os.environ["OPENAI_API_KEY"]}
    if OPENAI_BASE_URL:
        llm_kwargs["base_url"] = OPENAI_BASE_URL
    llm = ChatOpenAI(**llm_kwargs)
    smart_llm = with_memwal_langchain(
        llm,
        key=os.environ["MEMWAL_PRIVATE_KEY"],
        account_id=os.environ["MEMWAL_ACCOUNT_ID"],
        namespace=WALRUS_MEMORY_NAMESPACE,
        env=WALRUS_MEMORY_ENV,
        max_memories=5,
        min_relevance=0.2,
    )

    response = await smart_llm.ainvoke([
        HumanMessage(content="Summarize the customer's support context in one sentence.")
    ])
    print(response.content)
else:
    print("LangChain middleware demo skipped. Set RUN_MIDDLEWARE_DEMO=true to run it.")


## Basic Troubleshooting

- `401` or auth rejected: confirm the delegate private key belongs to `MEMWAL_ACCOUNT_ID`, and that both were created for the selected `staging` or `prod` environment.
- Invalid environment: this notebook only accepts `staging` and `prod`.
- Empty recall: wait for the remember job to finish, query the same namespace, and broaden the query or raise `max_distance`.
- Remember timeout: rerun `wait_for_remember_job(job_id, timeout_ms=...)` with a larger timeout.
- `Event loop is closed`: upgrade `memwal` and rerun the client cell. This notebook also resets the underlying HTTP client before each sync call for older releases.
- Optional SDK method skipped: the relayer may not have that feature enabled, or the current credentials may not have access.
- Middleware skipped: set `RUN_MIDDLEWARE_DEMO=true` and provide `OPENAI_API_KEY` through Colab Secrets or a hidden prompt.
- Secret safety: never print, paste into source cells, commit, or publish `MEMWAL_PRIVATE_KEY`. Rotate a delegate key if it was pasted into chat, logs, or a shared notebook output.


## Cleanup

Close the underlying HTTP client when you are done.


In [ ]:
memwal_call(memory.close)
